In [ ]:
%pylab inline
import datashader as ds
import pandas as pd
from astropy.cosmology import Planck18, z_at_value
import astropy.units as u
from datashader.utils import export_image

In [ ]:
import h5py

def extract_parttype(f, parttype):
    '''
    Extracts the particle IDs, coordinates and velocities of a given
    particle type from a snapshot file.

    Parameters:
    -----------
    f : h5py.File
        The snapshot file opened with h5py.
    parttype : int
        The particle type to extract. 1 for dark matter (halo), 2 for
        disk particles.

    Returns:
    --------
    np.ndarray of shape (N, 7)
        A 2D numpy array with the particle IDs, coordinates and velocities.
    '''
    I = f[f'PartType{parttype}/ParticleIDs'][:]  # Particle IDs
    C = f[f'PartType{parttype}/Coordinates'][:]  # Particle Coordinates
    V = f[f'PartType{parttype}/Velocities'][:]   # Particle Velocities
    return np.hstack((I[:, np.newaxis], C, V))

def open_snapshot(s):
    '''
    Opens a GADGET-4 snapshot file and extracts the dark matter and disk
    particle data from it.

    Parameters:
    -----------
    s : str
        The path to the snapshot file.
    '''
    with h5py.File(s, 'r') as f:
        G_halo = extract_parttype(f, parttype=1)
        return G_halo

In [ ]:
def age(redshift):
    return Planck18.age(redshift)
def useful(scale_param):
    return print(f"{1/scale_param-1}, {age(1/scale_param-1)}")

In [ ]:
def getdims(data): # used to determine the minimal and maximal occuring coordinates along each dimension
    for i in range(1,4):
        print(f"{min(data[:,i])} {max(data[:,i])}")
def getpandas(Data,idx): # creates a pandas dataframe from a dataset, omitting the selected coordinate (to later produce a 2D projection of the data)
    if idx==1:
        data=pd.DataFrame(Data[:,2:4])
        data.rename(columns={0: 'y', 1: 'z'}, inplace=True)
        return data
    elif idx==2:
        data=pd.DataFrame(column_stack(Data[:,1],Data[:,3]))
        data.rename(columns={0: 'x', 1: 'z'}, inplace=True)
        return data
    else:
        data=pd.DataFrame(Data[:,1:3])
        data.rename(columns={0: 'x', 1: 'y'}, inplace=True)
        return data

In [ ]:
data025=open_snapshot("data/snapshot_000.hdf5")
data050=open_snapshot("data/snapshot_001.hdf5")
data100=open_snapshot("data/snapshot_002.hdf5")

In [ ]:
useful(0.25)
useful(0.5)
useful(1)

In [ ]:
Size=500 # an appropriate size parameter for datashader

In [ ]:
ticks=linspace(0,Size,6).astype(int)
labels=linspace(0,50,6).astype(int).astype(str)

In [ ]:
fig, ax=plt.subplots(1,1,figsize=(6,6))

df=getpandas(data025,3)

cvs = ds.Canvas(plot_width=Size, plot_height=Size) # these 3 lines follow the pipeline on https://datashader.org/getting_started/Pipeline.html
agg = cvs.points(df,'x','y')
img=ds.tf.shade(agg)

ax.imshow(img.to_pil()) # I embed the image in a matplotlib plot, to have the usual control over axes, etc.
ax.set_xticks(ticks=ticks, labels=labels)
ax.set_yticks(ticks=flip(ticks), labels=labels)
ax.set_xlabel("x [Mpc/h]")
ax.set_ylabel("y [Mpc/h]")

savefig('data025n.png', dpi=70, bbox_inches='tight')

In [ ]:
fig, ax=plt.subplots(1,1,figsize=(6,6))

df=getpandas(data050,3)

cvs = ds.Canvas(plot_width=Size, plot_height=Size)
agg = cvs.points(df,'x','y')
img=ds.tf.shade(agg)

ax.imshow(img.to_pil())
ax.set_xticks(ticks=ticks, labels=labels)
ax.set_yticks(ticks=flip(ticks), labels=labels)
ax.set_xlabel("x [Mpc/h]")
ax.set_ylabel("y [Mpc/h]")

savefig('data050n.png', dpi=70, bbox_inches='tight')

In [ ]:
fig, ax=plt.subplots(1,1,figsize=(6,6))

df=getpandas(data100,3)

cvs = ds.Canvas(plot_width=Size, plot_height=Size)
agg = cvs.points(df,'x','y')
img=ds.tf.shade(agg)
#export_image(img, "img_025", fmt=".pdf")

ax.imshow(img.to_pil())
ax.set_xticks(ticks=ticks, labels=labels)
ax.set_yticks(ticks=flip(ticks), labels=labels)
ax.set_xlabel("x [Mpc/h]")
ax.set_ylabel("y [Mpc/h]")
#ax.set_title("a=1, $\;t_{\\text{age}}$=13.8 Gyr")

savefig('data100n.png', dpi=70, bbox_inches='tight')

In [ ]:
density2d1c=loadtxt("results2d1C.dat")
density2d1g=loadtxt("results2d1G.dat")
density2d6c=loadtxt("results2d6C.dat")
density2d6g=loadtxt("results2d6G.dat")
density3d1c=loadtxt("results3d1C.dat")
density3d1g=loadtxt("results3d1G.dat")
density3d6c=loadtxt("results3d6C.dat")
density3d6g=loadtxt("results3d6G.dat")

In [ ]:
    df=getpandas(data100,3)
    
    cvs = ds.Canvas(plot_width=Size, plot_height=Size)
    agg = cvs.points(df,'x','y')
    img=ds.tf.shade(agg)
    
    plt.imshow(img.to_pil())
    plt.xticks(ticks=ticks, labels=labels, fontsize=12)
    plt.yticks(ticks=flip(ticks), labels=labels, fontsize=10)
    plt.tick_params(axis='both', which='major', labelsize=10)
    plt.tick_params(axis='both', which='minor', labelsize=10)
    plt.xlabel("x [Mpc/h]", fontsize=10)
    plt.ylabel("y [Mpc/h]", fontsize=10)
savefig('data1.png', dpi=70, bbox_inches='tight')

In [ ]:
    plt.tick_params(axis='both', which='major', labelsize=12)
    plt.tick_params(axis='both', which='minor', labelsize=12)
    plt.imshow(log(flip(density2d1c.reshape(64,64).T,axis=0)+1), cmap="Blues")
    plt.xlabel("box number", fontsize=10)
    plt.ylabel("box number", fontsize=10)
    plt.colorbar()
    savefig('dens2d1c.png', dpi=70, bbox_inches='tight')

In [ ]:
    plt.tick_params(axis='both', which='major', labelsize=12)
    plt.tick_params(axis='both', which='minor', labelsize=12)
    plt.imshow(log(flip(density2d1g.reshape(64,64).T,axis=0)+1), cmap="Blues")
    plt.xlabel("box number", fontsize=10)
    plt.ylabel("box number", fontsize=10)
    plt.colorbar()
    savefig('dens2d1g.png', dpi=70, bbox_inches='tight')

In [ ]:
    plt.tick_params(axis='both', which='major', labelsize=12)
    plt.tick_params(axis='both', which='minor', labelsize=12)
    plt.imshow(flip(abs(density2d1c-density2d1g).reshape(64,64).T,axis=0), cmap="Blues")
    plt.xlabel("box number", fontsize=10)
    plt.ylabel("box number", fontsize=10)
    plt.colorbar()
    savefig('densdiff.png', dpi=70, bbox_inches='tight')

In [ ]:
plt.imshow(log(flip(abs(density2d1c.reshape(64,64)-density2d1g.reshape(64,64)).T,axis=0)+1))

In [ ]:
dft2dc=loadtxt("dft_results2d8C.dat")
dft2dg=loadtxt("dft_results2d8G.dat")
dft3dc=loadtxt("dft_results3d8C.dat")
dft3dg=loadtxt("dft_results3d8G.dat")
dif_dft=dft2dc-dft2dg

In [ ]:
def Abs(x):
    return sqrt(x[:,0]**2+x[:,1]**2)

In [ ]:
plt.tick_params(axis='both', which='major', labelsize=12)
plt.tick_params(axis='both', which='minor', labelsize=12)
plt.imshow(flip(Abs(dft2dc).reshape(64,64).T,axis=0), cmap="Blues")
#plt.imshow(log(flip(fft2dc.reshape(64,64).T,axis=0)+1), cmap="Blues")
plt.xlabel("frequency bin", fontsize=10)
plt.ylabel("frequency bin", fontsize=10)
plt.colorbar()
savefig('dft2dc.png', dpi=70, bbox_inches='tight')

In [ ]:
plt.tick_params(axis='both', which='major', labelsize=12)
plt.tick_params(axis='both', which='minor', labelsize=12)
plt.imshow(flip(Abs(dft2dc).reshape(64,64).T,axis=0), cmap="Blues")
#plt.imshow(log(flip(fft2dc.reshape(64,64).T,axis=0)+1), cmap="Blues")
plt.xlabel("frequency bin", fontsize=10)
plt.ylabel("frequency bin", fontsize=10)
plt.colorbar()
savefig('dft2dg.png', dpi=70, bbox_inches='tight')

In [ ]:
plt.tick_params(axis='both', which='major', labelsize=12)
plt.tick_params(axis='both', which='minor', labelsize=12)
plt.imshow(flip(Abs(dif_dft).reshape(64,64).T,axis=0), cmap="Blues")
#plt.imshow(log(flip(fft2dc.reshape(64,64).T,axis=0)+1), cmap="Blues")
plt.xlabel("frequency bin", fontsize=10)
plt.ylabel("frequency bin", fontsize=10)
plt.colorbar()
savefig('dif_dft.png', dpi=70, bbox_inches='tight')